# Phase 2 - DarkIR-lite Training

Fine-tunes DarkIR-m ("DarkIR-lite") on synthetic clean/dark EndoSLAM pairs.
`src/darkir_lite/model.py` + `train.py` were written against facts confirmed
in `notebooks/phase2a_explore` and validated locally, then smoke-tested here
(v5: `train_loss=0.0007, val_psnr=23.68, val_ssim=0.772` on a 20-step CPU
fallback run).

**GPU fix**: Kaggle's API-pushed kernels default to a Tesla P100 (compute
capability 6.0), and Kaggle's preinstalled torch (2.10.0) has zero Pascal
kernels -- every real CUDA op crashed. PyTorch dropped Pascal support in
2.8; versions 2.4-2.7 still support it, and DarkIR's own repo happens to
pin exactly `torch==2.5.1`/`torchvision==0.20.1`. This notebook now
reinstalls that exact pin -- a deliberate, documented exception to this
project's usual "never touch preinstalled torch on Kaggle" rule, justified
because the preinstalled build is already broken for the hardware Kaggle
assigns here.

**Resume support**: checks for `checkpoints_resume/darkir_lite_latest.pt`
(committed to the repo after any session that doesn't finish all 20
epochs) and resumes from it automatically if present.

Set `MAX_STEPS` below: a small number (e.g. `20`) for a quick
re-verification that the GPU fix actually works before spending real
quota, or `None` for the real full run.

## 0. Setup: clone our repo + DarkIR, reinstall Pascal-compatible torch, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream

%cd repo

# Deliberate exception to "never touch preinstalled torch on Kaggle" -- the
# preinstalled build (2.10.0) has zero Pascal (sm_60) kernels, so every real
# CUDA op fails on the P100 Kaggle assigns here regardless. torch 2.4-2.7
# still ship sm_60 kernels; DarkIR's own repo pins exactly this version, so
# installing it both fixes the P100 and matches what DarkIR was tested with.
# NOTE: --extra-index-url (not --index-url) -- --index-url REPLACES PyPI
# entirely, which breaks resolution of transitive deps like nvidia-cudnn-cu12
# that live on PyPI proper, not on pytorch.org's own wheel index.
!pip install -q torch==2.5.1 torchvision==0.20.1 --extra-index-url https://download.pytorch.org/whl/cu121

!pip install -q -r environment/requirements.txt

## 1. Resolve dataset mount + GPU check

In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# torch.cuda.is_available() only checks a driver is present, not that this
# build actually ships kernels for the assigned GPU's compute capability --
# that's exactly what went wrong before. Do a real op.
gpu_actually_usable = False
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    try:
        torch.zeros(1, device="cuda") + torch.zeros(1, device="cuda")
        gpu_actually_usable = True
        print("GPU FIX CONFIRMED: real CUDA op succeeded")
    except RuntimeError as e:
        print(f"GPU still not usable after torch reinstall: {e}")

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

## 2. Check for a resume checkpoint from a previous session

In [ ]:
RESUME_CANDIDATE = "checkpoints_resume/darkir_lite_latest.pt"
RESUME_PATH = RESUME_CANDIDATE if os.path.isfile(RESUME_CANDIDATE) else None
print(f"resume checkpoint: {RESUME_PATH or 'none found -- starting from the pretrained checkpoint'}")

## 3. Write a run-specific config (data.root filled in) and run train.py

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

RUN_CONFIG_PATH = "/kaggle/working/run_config.yaml"
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f)
print(f"wrote {RUN_CONFIG_PATH} with data.root = {DATA_ROOT}")

In [ ]:
MAX_STEPS = None  # real full run -- GPU fix confirmed in v7 (20 steps + partial val in ~14.5s on cuda)

cmd = [
    "python", "-m", "src.darkir_lite.train",
    "--config", "/kaggle/working/run_config.yaml",
    "--output-dir", "/kaggle/working/checkpoints",
]
if MAX_STEPS is not None:
    cmd += ["--max-steps", str(MAX_STEPS)]
if RESUME_PATH is not None:
    cmd += ["--resume", RESUME_PATH]

print("running:", " ".join(cmd))
import subprocess
result = subprocess.run(cmd)
assert result.returncode == 0, f"train.py exited with code {result.returncode}"

## 4. Confirm checkpoints were written

In [ ]:
import os
import torch

ckpt_dir = "/kaggle/working/checkpoints"
files = sorted(os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else []
print("checkpoint files:", files)
assert files, "expected at least one checkpoint file"

epoch_ckpts = sorted((f for f in files if f.startswith("epoch_")), key=lambda f: int(f.split("_")[1].split(".")[0]))
latest = os.path.join(ckpt_dir, epoch_ckpts[-1] if epoch_ckpts else sorted(files)[-1])
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
epoch = ckpt["epoch"]
print(f"loaded {latest}: epoch={epoch}, global_step={ckpt['global_step']}, "
      f"val_psnr={ckpt.get('val_psnr')}, val_ssim={ckpt.get('val_ssim')}")
print(f"\n{'TRAINING COMPLETE (epoch 19 reached)' if epoch >= 19 else f'session ended after epoch {epoch} -- resume needed to reach epoch 19'}")

## Done

**If `MAX_STEPS = 20`**: this was the GPU re-verification run. Check cell
output above for `GPU FIX CONFIRMED` (no CPU fallback) and note the
per-step timing to estimate the full run's wall-clock cost, then set
`MAX_STEPS = None` for the real run.

**If `MAX_STEPS = None`**: this was a real training attempt. If it printed
`TRAINING COMPLETE (epoch 19 reached)`, Phase 2 is done -- record final
PSNR/SSIM in `PROGRESS.md`. If it printed `session ended after epoch N`
(hit Kaggle's session limit or died silently), download the latest
`epoch_*.pt`, commit it to `checkpoints_resume/darkir_lite_latest.pt` in
the repo, push, and re-run this notebook -- it will auto-resume from
cell 2's check.